# BÀI TẬP: E-COMMERCE DATA (ONLINE RETAIL)
**Nguồn:** kaggle.com/datasets/carrie1/ecommerce-data (541,909 dòng)


## Setup

In [22]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from pathlib import Path
from scipy import stats

sns.set_style('whitegrid')

csv_path = 'https://raw.githubusercontent.com/databricks/Spark-The-Definitive-Guide/master/data/retail-data/all/online-retail-dataset.csv'

df = pd.read_csv(csv_path, encoding='ISO-8859-1', on_bad_lines='skip')
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
print('Loaded from:', csv_path)
df.head()

Loaded from: https://raw.githubusercontent.com/databricks/Spark-The-Definitive-Guide/master/data/retail-data/all/online-retail-dataset.csv


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


---
# PHẦN A — DATA PROFILING
## A.1. Data size, column names, data types

In [23]:
# TODO
print("data size:", df.shape)
print("columns:", df.columns.tolist())
print("Dtypes:", df.dtypes)

data size: (541909, 8)
columns: ['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country']
Dtypes: InvoiceNo                 str
StockCode                 str
Description               str
Quantity                int64
InvoiceDate    datetime64[us]
UnitPrice             float64
CustomerID            float64
Country                   str
dtype: object


## A.2. Missing values & Duplicate data

In [24]:
# TODO
print("Missing values:\n", df.isnull().sum())
print("Duplicates:", df.duplicated().sum())

Missing values:
 InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64
Duplicates: 5268


## A.3. Invalid values

In [25]:
# TODO
print("Quantity <= 0:", (df['Quantity'] <= 0).sum())
print("UnitPrice <= 0:", (df['UnitPrice'] <= 0).sum())

Quantity <= 0: 10624
UnitPrice <= 0: 2517


## A.4. Create a new column
Làm sạch dữ liệu (loại Quantity<=0, UnitPrice<=0), tạo cột `Sales` = Quantity * UnitPrice.

In [26]:
# TODO
df_clean = df[(df['Quantity'] > 0) & (df['UnitPrice'] > 0)].copy()
df_clean['Sales'] = df_clean['Quantity'] * df_clean['UnitPrice']
df_clean.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Sales
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,15.30
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34


---
# PHẦN B — DESCRIPTIVE STATISTICS
## Group 1 — Central Tendency

In [27]:
# TODO
print("Mean:", df_clean[['Quantity', 'UnitPrice', 'Sales']].mean())
print("Median:", df_clean[['Quantity', 'UnitPrice', 'Sales']].median())

Mean: Quantity     10.542037
UnitPrice     3.907625
Sales        20.121871
dtype: float64
Median: Quantity     3.00
UnitPrice    2.08
Sales        9.90
dtype: float64


## Group 2 — Dispersion

In [28]:
# TODO
print("Standard Deviation:\n", df_clean[['Quantity', 'UnitPrice', 'Sales']].std())

Standard Deviation:
 Quantity     155.524124
UnitPrice     35.915681
Sales        270.356743
dtype: float64


## Group 3 — Location and Shape

In [29]:
# TODO
print("Skewness:\n", df_clean[['Quantity', 'UnitPrice', 'Sales']].skew())

Skewness:
 Quantity     471.727716
UnitPrice    206.087555
Sales        506.706012
dtype: float64


---
# PHẦN C — DEFINE THE QUESTION

## Câu hỏi 1: Quốc gia nào đóng góp doanh thu cao nhất, chiếm bao nhiêu % tổng doanh thu?

In [30]:
# TODO
total_sales = df_clean['Sales'].sum()
country_sales = df_clean.groupby('Country')['Sales'].sum().sort_values(ascending=False)
country_pct = (country_sales / total_sales) * 100

summary_country = pd.DataFrame({'Sales': country_sales, 'Percentage (%)': country_pct})
print(summary_country.head())

                      Sales  Percentage (%)
Country                                    
United Kingdom  9025222.084       84.611315
Netherlands      285446.340        2.676055
EIRE             283453.960        2.657376
Germany          228867.140        2.145626
France           209715.110        1.966076


## Câu hỏi 2: Sản phẩm nào bán chạy nhất theo doanh thu?

In [31]:
# TODO
top_products = df_clean.groupby(['StockCode', 'Description'])['Sales'].sum().reset_index()
top_products = top_products.sort_values(by='Sales', ascending=False)
print("Top 5 sản phẩm có doanh thu cao nhất:")
print(top_products.head())

Top 5 sản phẩm có doanh thu cao nhất:
     StockCode                         Description      Sales
4150       DOT                      DOTCOM POSTAGE  206248.77
1340     22423            REGENCY CAKESTAND 3 TIER  174484.74
2668     23843         PAPER CRAFT , LITTLE BIRDIE  168469.60
3640    85123A  WHITE HANGING HEART T-LIGHT HOLDER  104340.29
2877     47566                       PARTY BUNTING   99504.33


## Câu hỏi 3: Doanh số có tính mùa vụ theo tháng không?

In [32]:
# TODO
df_clean['Month'] = df_clean['InvoiceDate'].dt.to_period('M')
monthly_sales = df_clean.groupby('Month')['Sales'].sum()
print(monthly_sales)

Month
2010-12     823746.140
2011-01     691364.560
2011-02     523631.890
2011-03     717639.360
2011-04     537808.621
2011-05     770536.020
2011-06     761739.900
2011-07     719221.191
2011-08     759138.380
2011-09    1058590.172
2011-10    1154979.300
2011-11    1509496.330
2011-12     638792.680
Freq: M, Name: Sales, dtype: float64


## Câu hỏi 4: Giá trị đơn hàng trung bình (Average Order Value) khác nhau thế nào giữa các quốc gia?

In [33]:
# TODO
order_values = df_clean.groupby(['Country', 'InvoiceNo'])['Sales'].sum().reset_index()
aov_by_country = order_values.groupby('Country')['Sales'].mean().sort_values(ascending=False)
print(aov_by_country.head(10))

Country
Singapore      3039.898571
Netherlands    3036.663191
Australia      2430.198421
Japan          1969.282632
Lebanon        1693.880000
Hong Kong      1426.527273
Brazil         1143.600000
Sweden         1066.064722
Switzerland    1057.220370
Denmark        1053.074444
Name: Sales, dtype: float64


## Câu hỏi 5: Tỷ lệ giao dịch có dấu hiệu trả hàng/hủy (Quantity âm ở dữ liệu gốc) khác nhau thế nào giữa các quốc gia?

In [34]:
# TODO
cancels = df[df['Quantity'] < 0].groupby('Country')['InvoiceNo'].count()
total_tx = df.groupby('Country')['InvoiceNo'].count()
cancel_rate = (cancels / total_tx * 100).sort_values(ascending=False).dropna()
print("Tỷ lệ hủy/trả hàng theo quốc gia (%):\n", cancel_rate.head(10))

Tỷ lệ hủy/trả hàng theo quốc gia (%):
 Country
USA               38.487973
Czech Republic    16.666667
Malta             11.811024
Japan             10.335196
Saudi Arabia      10.000000
Australia          5.877681
Italy              5.603985
Bahrain            5.263158
Germany            4.770932
EIRE               3.684724
Name: InvoiceNo, dtype: float64


## Câu hỏi 6 (Tổng hợp) — Viết insight tổng hợp
Dựa trên Phần A, B, C, viết 4-5 câu insight tổng thể.

*(Viết insight của bạn vào đây...)*